# HS4002 Week 6

## OLS with Higher-Order Effects, Interaction Terms, and Logistic Regression

Today we cover:
1. Polynomial terms in OLS (age-squared)
2. Interaction effects
3. The Linear Probability Model (LPM)
4. Logistic regression
5. Marginal effects


In [1]:
# !pip install pandas numpy plotnine statsmodels marginaleffects stargazer

import pandas as pd
import numpy as np
from plotnine import *
import warnings
warnings.filterwarnings('ignore')
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm

In [2]:
raw_df = pd.read_csv("gss2022_mini.csv")

chosen_variables = [
	'id', 'age', 'yrlvmus', 'yrartxbt', 'yrmovie', 'yrcreat',
	'income16', 'prestg10', 'degree', 'race', 'sex', 'wrkstat'
]
df = raw_df[chosen_variables].dropna().copy()

# Recode income16 to continuous midpoint values
income_map = {
	1: 500,    2: 2000,   3: 3500,   4: 4500,   5: 5500,   6: 6500,
	7: 7500,   8: 9000,   9: 11250,  10: 13750, 11: 16250, 12: 18750,
	13: 21250, 14: 23750, 15: 27500, 16: 32500, 17: 37500, 18: 45000,
	19: 55000, 20: 67500, 21: 82500, 22: 100000, 23: 120000, 24: 140000,
	25: 160000, 26: 250000
}
df['income_cont'] = df['income16'].map(income_map)

# Binary cultural participation indicators
df['binary_lvmus']  = (df['yrlvmus']  == 1).astype(int)
df['binary_artxbt'] = (df['yrartxbt'] == 1).astype(int)
df['binary_movie']  = (df['yrmovie']  == 1).astype(int)
df['binary_creat']  = (df['yrcreat']  == 1).astype(int)
df['omni'] = df[['binary_lvmus','binary_artxbt','binary_movie','binary_creat']].sum(axis=1)

# Derived variables
df['race_bin']   = df['race'].astype(str)
df['ba_binary']  = (df['degree'] >= 3).astype(int)
df['sex_woman']  = np.where(df['sex'] == 2, 'Woman', 'Not Woman')
df['working']    = np.where(df['wrkstat'] <= 3, 'Working', 'Not Working')

# OLS with Higher-Order Polynomials

Begin by regressing income on age. Save as `ols_model1`.

In [3]:
ols_model1 = smf.ols('income_cont ~ age', data=df).fit()
ols_model1.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:            income_cont   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                 -0.001
Method:                 Least Squares   F-statistic:                   0.01847
Date:                Wed, 16 Sep 2026   Prob (F-statistic):              0.892
Time:                        11:35:38   Log-Likelihood:                -8910.5
No. Observations:                 707   AIC:                         1.782e+04
Df Residuals:                     705   BIC:                         1.783e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   8.632e+04   8546.726     10.099      0.000    6.95e+04    1.03e+05
age           21.2431    156.330      0.136      0.892    -285.684     328.171
==============================================================================
Omnibus:                      113.766   Durbin-Watson:                   1.629
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              168.872
Skew:                           1.178   Prob(JB):                     2.14e-37
Kurtosis:                       3.430   Cond. No.                         172.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

## Adding an age-squared term

In Python formulas, use `I(age**2)` — the `I()` wrapper tells the formula parser to treat the expression literally.

In [4]:
ols_model2 = smf.ols('income_cont ~ age + I(age**2)', data=df).fit()
ols_model2.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:            income_cont   R-squared:                       0.021
Model:                            OLS   Adj. R-squared:                  0.018
Method:                 Least Squares   F-statistic:                     7.568
Date:                Wed, 16 Sep 2026   Prob (F-statistic):           0.000560
Time:                        11:35:49   Log-Likelihood:                -8903.0
No. Observations:                 707   AIC:                         1.781e+04
Df Residuals:                     704   BIC:                         1.783e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
===============================================================================
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept   -2088.6030   2.43e+04     -0.086      0.931   -4.97e+04    4.55e+04
age          3886.3291   1006.049      3.863      0.000    1911.113    5861.545
I(age ** 2)   -37.4677      9.636     -3.888      0.000     -56.387     -18.548
==============================================================================
Omnibus:                      109.972   Durbin-Watson:                   1.611
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              160.918
Skew:                           1.148   Prob(JB):                     1.14e-35
Kurtosis:                       3.440   Cond. No.                     3.16e+04
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 3.16e+04. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

## Making a regression table

As in Week 5, the `stargazer` package collects fitted models into a single regression table. Reading models side by side is much easier than scrolling through two separate `.summary()` blocks.

In [5]:
from stargazer.stargazer import Stargazer

sg = Stargazer([ols_model1, ols_model2])
sg.title('Income, Age, and Age-Squared')

# Label the models and variables readably, rather than by raw column name
sg.dependent_variable_name('Income (USD)')

sg.custom_columns(['Model 1', 'Model 2'], [1, 1])
sg.covariate_order(['age', 'I(age ** 2)', 'Intercept'])
sg.rename_covariates({
	'age':         'Age',
	'I(age ** 2)': 'Age-squared',
	'Intercept':   'Intercept'
})

# Ending the cell on the object itself renders the table in the notebook
sg

In [10]:
person_a = 15
person_b = 45
#expected difference in model 2 , expected difference is:
(person_b - person_a)*3886 + ((person_b - person_a)**2) * (-37.5)
# downward sloping curve
# writing it out in functional form 
# the number means ...dependent is income, expected income difference

82830.0

## Interaction Effects

Add BA education and gender to the model. Save as `ols_model3`.

In [13]:
ols_model3 = smf.ols('income_cont ~ age + I(age**2) + ba_binary + sex_woman', data=df).fit()
ols_model3.summary()
#added variable someone having bachelors degree 

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:            income_cont   R-squared:                       0.203
Model:                            OLS   Adj. R-squared:                  0.199
Method:                 Least Squares   F-statistic:                     44.79
Date:                Wed, 16 Sep 2026   Prob (F-statistic):           1.61e-33
Time:                        11:45:08   Log-Likelihood:                -8830.1
No. Observations:                 707   AIC:                         1.767e+04
Df Residuals:                     702   BIC:                         1.769e+04
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
======================================================================================
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept          -8893.2601   2.21e+04     -0.402      0.688   -5.23e+04    3.46e+04
sex_woman[T.Woman] -1.663e+04   4871.657     -3.414      0.001   -2.62e+04   -7067.441
age                 3275.5520    910.156      3.599      0.000    1488.599    5062.505
I(age ** 2)          -31.3456      8.720     -3.595      0.000     -48.466     -14.225
ba_binary            6.04e+04   4874.146     12.393      0.000    5.08e+04       7e+04
==============================================================================
Omnibus:                      102.337   Durbin-Watson:                   1.765
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              145.602
Skew:                           1.046   Prob(JB):                     2.42e-32
Kurtosis:                       3.751   Cond. No.                     3.19e+04
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 3.19e+04. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

Now add an interaction term between BA and gender. In statsmodels formulas, `:` creates an interaction.

Save as `ols_model4`.

In [14]:
ols_model4 = smf.ols(
	'income_cont ~ age + I(age**2) + ba_binary + sex_woman + ba_binary:sex_woman',
	data=df
).fit()

In [15]:
sg = Stargazer([ols_model3, ols_model4])
sg.title('Interaction Between BA and Gender')
sg.dependent_variable_name('Income (USD)')

sg.custom_columns(['Model 3', 'Model 4'], [1, 1])
sg.covariate_order([
	'age',
	'I(age ** 2)',
	'ba_binary',
	'sex_woman[T.Woman]',
	'ba_binary:sex_woman[T.Woman]',
	'Intercept'
])
sg.rename_covariates({
	'age':                          'Age',
	'I(age ** 2)':                  'Age-squared',
	'ba_binary':                    'BA degree',
	'sex_woman[T.Woman]':           'Gender (woman = 1)',
	'ba_binary:sex_woman[T.Woman]': 'BA × Woman',
	'Intercept':                    'Intercept'
})

sg

In [16]:
# BA degree as binary variable, when BA=1, you have it. Model 3 60403 for BA tells us the expected diffrence betw someone w BA and no BA (earnning 60k more)
# gender has no interaction in model 3 and model 3 shows that expected difference between being woman and not a woman is that women will make on average -16k 
# model 4 adds the interaction of BA x woman , differnece between men and women not just abt gender, but also the payoff of education between men vs women
# the interaction term is an effect between having bachelors and being a woman ie in this case, BA and woman, u earn -17k
# applies on top of the main effect 
# understand what the dependent variable is and how independent variables are coded 



# Using the Linear Probability Model (LPM)

The LPM uses OLS on a binary dependent variable. It's quick and interpretable, though it can predict probabilities outside [0, 1].

Regress live music attendance (`binary_lvmus`) on BA education and log income.

In [17]:
lpm_model1 = smf.ols(
	'binary_lvmus ~ ba_binary + np.log(income_cont)',
	data=df
).fit()
print(lpm_model1.summary())

                            OLS Regression Results                            
Dep. Variable:           binary_lvmus   R-squared:                       0.069
Model:                            OLS   Adj. R-squared:                  0.066
Method:                 Least Squares   F-statistic:                     26.08
Date:                Wed, 16 Sep 2026   Prob (F-statistic):           1.19e-11
Time:                        11:56:45   Log-Likelihood:                -486.80
No. Observations:                 707   AIC:                             979.6
Df Residuals:                     704   BIC:                             993.3
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
Intercept              -0.5077    

In [20]:
# logincome increases by one unit, likelihood of gg concert increases by 0.088
# income now coded as the log of it, so interpretation has to be done in that light 


# Using a Logit Model

Now, let's try logistic regression.

In [21]:
logit_model1 = smf.logit(
	'binary_lvmus ~ ba_binary + np.log(income_cont)',
	data=df
).fit()
print(logit_model1.summary())

Optimization terminated successfully.
         Current function value: 0.655998
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:           binary_lvmus   No. Observations:                  707
Model:                          Logit   Df Residuals:                      704
Method:                           MLE   Df Model:                            2
Date:                Wed, 16 Sep 2026   Pseudo R-squ.:                 0.05151
Time:                        11:57:39   Log-Likelihood:                -463.79
converged:                       True   LL-Null:                       -488.98
Covariance Type:            nonrobust   LLR p-value:                 1.151e-11
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
Intercept              -4.4595      0.961     -4.639      0.000      -6.344      -2.575
ba

In [22]:
# 0.5186 is the expected incr in the odds of going to a concert wrt to someone having a BA as compared to someone without one 


Build a second logit model adding gender, age, and age-squared. Compare both.

In [23]:
logit_model2 = smf.logit(
	'binary_lvmus ~ ba_binary + np.log(income_cont) + sex_woman + age + I(age**2)',
	data=df
).fit()
print(logit_model2.summary())

Optimization terminated successfully.
         Current function value: 0.645323
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:           binary_lvmus   No. Observations:                  707
Model:                          Logit   Df Residuals:                      701
Method:                           MLE   Df Model:                            5
Date:                Wed, 16 Sep 2026   Pseudo R-squ.:                 0.06695
Time:                        11:58:17   Log-Likelihood:                -456.24
converged:                       True   LL-Null:                       -488.98
Covariance Type:            nonrobust   LLR p-value:                 8.952e-13
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
Intercept              -3.6710      1.156     -3.176      0.001      -5.936      -1.405
se

In [24]:
# compare lpm and logit models using stargazer
sg = Stargazer([lpm_model1, logit_model1, logit_model2])
sg.title('Live Music Attendance: LPM vs Logit')
sg.dependent_variable_name('Attends live music')

sg.custom_columns(['LPM', 'Logit 1', 'Logit 2'], [1, 1, 1])
sg.covariate_order([
	'ba_binary',
	'np.log(income_cont)',
	'sex_woman[T.Woman]',
	'age',
	'I(age ** 2)',
	'Intercept'
])
sg.rename_covariates({
	'ba_binary':           'BA degree',
	'np.log(income_cont)': 'log(Income)',
	'sex_woman[T.Woman]':  'Gender (woman = 1)',
	'age':                 'Age',
	'I(age ** 2)':         'Age-squared',
	'Intercept':           'Intercept'
})

# The LPM coefficients are changes in probability, the logit coefficients are
# log-odds, so compare signs and significance across columns — not magnitudes.
sg



In [26]:
# explain them in words, thats to understand
# understand the different models are important - they are all sharing some similiarities but also have some differences 
# understand how they are similiar and how they are different

## Comparing Logit Models

### AIC comparison

In [27]:
print(f'AIC logit_model1: {logit_model1.aic:.2f}')
print(f'AIC logit_model2: {logit_model2.aic:.2f}')

AIC logit_model1: 933.58
AIC logit_model2: 924.49


In [28]:
# AIC is Akaike Information Criterion - smaller AIC is better 

### Likelihood ratio test


In [29]:
from scipy.stats import chi2

lr_stat = 2 * (logit_model2.llf - logit_model1.llf)
df_diff = logit_model2.df_model - logit_model1.df_model
p_value = chi2.sf(lr_stat, df_diff)
print(f'LR statistic = {lr_stat:.4f}')
print(f'df = {df_diff:.0f}')
print(f'p-value = {p_value:.4f}')

LR statistic = 15.0945
df = 3
p-value = 0.0017


In [31]:
# Compares two models within each other and whether they are statistically significant and then reject null in this case
# the null in this case is that other variables all have a beta coeff of 0

# Marginal Effects

The Python `marginaleffects` package mirrors R's package of the same name.

In [33]:
%pip install marginaleffects
from marginaleffects import avg_comparisons, comparisons

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 11.7 MB/s  0:00:00 13.2 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.2/63.2 MB 8.6 MB/s  0:00:07 eta 0:00:010:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 562.6/562.6 kB 9.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 865.8/865.8 kB 12.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 MB 6.4 MB/s  0:00:06 eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.9/35.9 MB 2.2 MB/s  0:00:16 eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 2.5 MB/s  0:00:002.3 MB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16/16 [marginaleffects] 13/16 [jax]rs]-32]
Note: you may need to restart the kernel to use updated packages.


## Average marginal effect

The average marginal effect of a BA degree — equivalent to R's `avg_comparisons(logit_model1, variables='ba_binary')`.

In [34]:
avg_comparisons(logit_model1, variables='ba_binary')

term,contrast,estimate,std_error,statistic,p_value,s_value,conf_low,conf_high
str,str,f64,f64,f64,f64,f64,f64,f64
"""ba_binary""","""1 - 0""",0.123828,0.040752,3.03855,0.002465,8.664096,0.043817,0.203839


Interpretation: on average across our sample, having a BA degree increases the **probability** of attending a live music concert by X percentage points, holding other variables constant.

## Marginal effect at the mean

The marginal effect for a hypothetical person at the average of all X variables.

In [35]:
comparisons(logit_model1, variables='ba_binary', newdata='mean')

term,contrast,estimate,std_error,statistic,p_value,s_value,conf_low,conf_high
str,str,f64,f64,f64,f64,f64,f64,f64
"""ba_binary""","""1 - 0""",0.126466,0.041026,3.082541,0.002133,8.873215,0.045917,0.207014


In [37]:
# look at variables and build your own before submitting this lab

In [38]:
# My Model will calculate how much education, income and gender affect the odds that a person attends an art exihbit
logit_my_model = smf.logit('binary_artxbt ~ ba_binary + np.log(income_cont) + sex_woman', data=df).fit()

Optimization terminated successfully.
         Current function value: 0.622848
         Iterations 5


In [39]:
print(logit_my_model.summary())

                           Logit Regression Results                           
Dep. Variable:          binary_artxbt   No. Observations:                  707
Model:                          Logit   Df Residuals:                      703
Method:                           MLE   Df Model:                            3
Date:                Thu, 17 Sep 2026   Pseudo R-squ.:                 0.03476
Time:                        09:39:54   Log-Likelihood:                -440.35
converged:                       True   LL-Null:                       -456.21
Covariance Type:            nonrobust   LLR p-value:                 6.019e-07
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
Intercept              -3.6129      1.020     -3.544      0.000      -5.611      -1.615
sex_woman[T.Woman]      0.2277      0.164      1.392      0.164      -0.093       0.548
ba_binary       

In [42]:
# Holding income and gender constant, 0.5842 is the expected increase in the log odds of going to an art exhibit for someone with a BA as compared to someone without one
# Holding gender and education constant, 0.2320 is the expected increase in log odds of attending an art exhibit for an one unit increase in income
# for income and gender being held constant, this relationship is statistically significant, with p = 0.01 which is less than 0.05, thus rejecting the null hypothesis and therefore, BA has a statistically significant  effect on goinng to an art exhbit
# holding gender and education constant, this relationship is statistically significant, with p = 0.013, which is less than 0.05, thus we reject the null hypothesis and conclude that log income has an effect on the chances going for an art exhibit 
# holding education and income constant, 0.2277 is the expected change in log odds of attending an art exhbit for a woman as compared to a non-woman
# since the p value in this case is 0.164 which is more than 0.05, we do not reject the null hypothesis and conclude there is no statistically significant difference between the chances of going for an art exhbit between a woman and non woman


In [47]:
print(f'AIC logit_my_model: {logit_my_model.aic:.2f}')

AIC logit_my_model: 888.71


In [48]:
print(f'AIC logit_model1: {logit_model1.aic:.2f}')
print(f'AIC logit_model2: {logit_model2.aic:.2f}')
print(f'AIC logit_my_model: {logit_my_model.aic:.2f}')

AIC logit_model1: 933.58
AIC logit_model2: 924.49
AIC logit_my_model: 888.71


In [49]:
# The AIC is smaller in my model, therefore indicating a better fit to the data